# Custom bars with `agg` dictionaries

`resample` reduces a stream into bars. Its `agg` argument accepts a string
shorthand for common cases (`"first"`, `"last"`, `"min"`, `"max"`, `"sum"`,
`"count"`, `"mean"`, `"ohlc"`, `"ohlcv"`, `"ohlcv2"`), or a dictionary
`{name: reducer}` to build your own labelled bars.

This recipe builds OHLC price bars with separate buy and sell volume from a
stream of trades, using the dictionary form.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from screamer import PosPart, NegPart
from screamer.streams import resample

rng = np.random.default_rng(7)
n = 400
price = 100 + np.cumsum(rng.normal(size=n))
signed_vol = rng.normal(size=n) * rng.integers(1, 5, size=n)   # buys positive, sells negative
t = np.arange(n, dtype=np.int64)
BAR = 40

Open, high, low, and close come from the price stream through a dictionary of
`first`, `max`, `min`, and `last`. Buy and sell volume come from the signed
volume: its positive part summed per bar for buys, its negative part for sells.

In [ ]:
ohlc = resample(
    price, t, every=BAR,
    agg={"open": "first", "high": "max", "low": "min", "close": "last"},
)
buy = resample(np.asarray(PosPart()(signed_vol)), t, every=BAR, agg="sum")
sell = resample(np.asarray(NegPart()(signed_vol)), t, every=BAR, agg="sum")

o, h, l, c = ohlc["open"], ohlc["high"], ohlc["low"], ohlc["close"]
buy_vol, sell_vol = buy.values, sell.values

In [ ]:
fig = plt.figure(figsize=(10, 6))
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.08)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

x = np.arange(len(o))
for i in range(len(x)):
    color = "green" if c[i] >= o[i] else "red"
    ax1.plot([x[i], x[i]], [l[i], h[i]], color=color, lw=1)   # wick
    ax1.plot([x[i], x[i]], [o[i], c[i]], color=color, lw=6)   # body
ax1.set_ylabel("price")
ax1.set_title("OHLC bars with buy and sell volume")

ax2.bar(x, buy_vol, color="green", width=0.6)
ax2.bar(x, -sell_vol, color="red", width=0.6)
ax2.axhline(0, color="k", lw=0.5)
ax2.set_ylabel("volume")
ax2.set_xlabel("bar")
plt.tight_layout()

The dictionary form is open-ended: add another key, such as a per-bar
`ExpandingStd()` for volatility, to grow the bar schema.